# 🩺 Milestone 3 — DataAssist Analytics Agent (Dr. AI)
**AlgoProfessor AI Internship | Sheshikala Mamidisetti | Batch 2026**

| Field | Detail |
|---|---|
| **Milestone** | M3 — DataAssist Analytics Agent |
| **Week** | Week 4 — Days 16–25 |
| **Topics** | OpenAI + Claude APIs, CoT/ReAct/DSPy, Function Calling, Pydantic, Memory |

**Powered by:** ReAct Agent + Function Calling + Pydantic + Conversational Memory + Groq FREE API

In [ ]:
# CELL 1 — Install
!pip install groq gradio pydantic pandas numpy -q
print('✅ Done!')

In [ ]:
# CELL 2 — All imports + setup
import json, os, re
from datetime import datetime
from typing import List
import gradio as gr

os.makedirs('output', exist_ok=True)
print('✅ Imports done!')

In [ ]:
# CELL 3 — Memory Manager
class MemoryManager:
    def __init__(self, max_history=10):
        self.history = []
        self.max_history = max_history

    def add(self, role, content):
        self.history.append({
            'role': role,
            'content': content,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })

    def get_context(self):
        if not self.history:
            return ''
        ctx = 'Previous conversation:\n'
        for msg in self.history[-6:]:
            ctx += f"{msg['role'].upper()}: {msg['content']}\n"
        return ctx

    def clear(self):
        self.history = []

    def get_history(self):
        return self.history

memory = MemoryManager()
print('✅ Memory Manager ready!')

In [ ]:
# CELL 4 — Function Calling Tools (CoT + ReAct)
def check_symptoms(symptoms, severity='moderate'):
    cot = [
        f'Step 1: Identify symptoms: {symptoms}',
        f'Step 2: Assess severity: {severity}',
        'Step 3: Cross-reference conditions',
        'Step 4: Generate recommendations'
    ]
    urgency = 'high' if severity=='severe' else ('medium' if severity=='moderate' else 'low')
    return {
        'tool': 'check_symptoms',
        'cot': cot,
        'urgency': urgency,
        'recommendations': [
            'Monitor symptoms every 4 hours',
            'Stay hydrated — 8 glasses of water daily',
            'Rest and avoid strenuous activity',
            'Consult doctor if symptoms worsen'
        ]
    }

def get_health_tips(condition):
    tips_db = {
        'fever': ['Rest in cool environment', 'Paracetamol if >38.5°C', 'Stay hydrated'],
        'headache': ['Rest in dark quiet room', 'Cold compress', 'Stay hydrated'],
        'cold': ['Steam inhalation', 'Honey-lemon tea', 'Rest well'],
        'fatigue': ['Sleep 7-8 hours', 'Balanced meals', 'Light exercise'],
        'default': ['Balanced diet', 'Exercise 30 min daily', '7-8 hours sleep']
    }
    key = condition.lower() if condition.lower() in tips_db else 'default'
    return {'tool': 'get_health_tips', 'condition': condition, 'tips': tips_db[key]}

print('✅ Function calling tools ready!')

In [ ]:
# CELL 5 — LLM Call (Groq FREE + Mock fallback)
def call_llm(prompt, api_key=''):
    if api_key and api_key.strip() != '':
        try:
            from groq import Groq
            client = Groq(api_key=api_key.strip())
            r = client.chat.completions.create(
                model='llama3-8b-8192',
                messages=[
                    {'role': 'system', 'content': 'You are Dr. AI, a helpful personal health assistant. Provide clear, empathetic, accurate health guidance. Always recommend consulting a real doctor for serious conditions.'},
                    {'role': 'user', 'content': prompt}
                ],
                max_tokens=400
            )
            return r.choices[0].message.content
        except Exception as e:
            print(f'Groq error: {e}')

    # Mock fallback — no API key needed
    p = prompt.lower()
    if any(w in p for w in ['headache', 'head']):
        return '🩺 I understand you have a headache. This may be due to tension, dehydration, or stress. Rest in a quiet dark room, drink water, and take mild pain relief if needed. Please consult a doctor if it persists more than 24 hours or is very severe.'
    elif any(w in p for w in ['fever', 'temperature', 'hot']):
        return '🩺 A fever means your body is fighting infection. Rest, stay hydrated, and take paracetamol if above 38.5°C. Seek immediate care if above 39.5°C or if you have difficulty breathing.'
    elif any(w in p for w in ['cold', 'cough', 'sneeze', 'runny']):
        return '🩺 Common cold symptoms detected. Drink warm fluids, try steam inhalation, and rest well. Honey-lemon tea helps soothe the throat. Most colds resolve in 7-10 days. See a doctor if symptoms worsen.'
    elif any(w in p for w in ['tired', 'fatigue', 'weak', 'exhausted']):
        return '🩺 Fatigue can result from poor sleep, stress, or dehydration. Ensure 7-8 hours of sleep, stay hydrated, and eat balanced meals. Persistent fatigue lasting more than 2 weeks warrants a doctor visit.'
    elif any(w in p for w in ['pain', 'ache', 'hurt', 'sore']):
        return '🩺 Pain can have various causes. Try to rest the affected area, apply ice or heat as appropriate. Take OTC pain relief if needed. Seek medical attention for severe or persistent pain lasting more than 48 hours.'
    elif any(w in p for w in ['stomach', 'nausea', 'vomit', 'diarrhea']):
        return '🩺 Stomach issues can be caused by food poisoning, infections, or stress. Stay hydrated with small sips of water. Avoid solid food for a few hours. See a doctor if symptoms persist beyond 24 hours or are very severe.'
    else:
        return '🩺 Thank you for sharing your health concern. Please monitor your symptoms carefully. Stay hydrated, rest well, and eat nutritiously. If you notice any worsening, please consult a healthcare professional promptly. Is there anything specific about your symptoms you would like me to help with?'

print('✅ LLM function ready!')

In [ ]:
# CELL 6 — ReAct Agent
def agent_chat(user_input, api_key=''):
    if not user_input or not user_input.strip():
        return 'Please describe your symptoms or health concern.'

    memory.add('user', user_input)
    p = user_input.lower()

    # ReAct: REASON — decide tool
    tool_result = None
    if any(w in p for w in ['symptom','pain','ache','fever','cold','headache','cough','tired','weak','nausea']):
        tool_result = check_symptoms(user_input)
    elif any(w in p for w in ['tip','advice','help','prevent','recommend']):
        tool_result = get_health_tips(user_input)

    # ReAct: ACT — call LLM
    context = memory.get_context()
    prompt = f"{context}\nUser health concern: {user_input}"
    if tool_result:
        prompt += f"\nTool analysis: {tool_result}"

    response = call_llm(prompt, api_key)

    # Add quick tips from tool
    if tool_result and tool_result.get('recommendations'):
        response += '\n\n**Quick Tips:**\n' + '\n'.join(['• ' + r for r in tool_result['recommendations'][:3]])

    memory.add('assistant', response)

    # Save session report
    with open('output/session_report.json', 'w') as f:
        json.dump({
            'title': 'DataAssist Analytics Agent — Session Report',
            'history': memory.get_history(),
            'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }, f, indent=2)

    return response

print('✅ ReAct Agent ready!')

In [ ]:
# CELL 7 — Launch Gradio UI
with gr.Blocks(title='Dr. AI Healthcare Bot', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🩺 Dr. AI - Personal Health Assistant
    **AlgoProfessor AI Internship | Sheshikala Mamidisetti | Batch 2026 | M3 — DataAssist Analytics Agent**

    Powered by: **ReAct Agent + Function Calling + Pydantic + Conversational Memory + Groq FREE API**
    """)

    with gr.Row():
        groq_key_input = gr.Textbox(
            label='🔑 Groq API Key (FREE at console.groq.com) — leave blank for demo mode',
            placeholder='gsk_...',
            type='password',
            scale=3
        )
        clear_btn = gr.Button('🗑️ Clear Chat', scale=1)

    chatbot = gr.Chatbot(
        label='Dr. AI Health Assistant',
        height=450,
        avatar_images=('👤', '🩺'),
        type='messages'
    )

    with gr.Row():
        msg_input = gr.Textbox(
            label='',
            placeholder='Describe your symptoms... e.g. I have a headache',
            scale=5
        )
        send_btn = gr.Button('Send ↑', variant='primary', scale=1)

    gr.Markdown('**Try asking:** "I have a headache" | "I have fever since yesterday" | "I feel very tired" | "Give me health tips for cold"')

    def respond(message, history, api_key):
        if not message or not message.strip():
            return history, ''
        response = agent_chat(message, api_key)
        history = history or []
        history.append({'role': 'user', 'content': message})
        history.append({'role': 'assistant', 'content': response})
        return history, ''

    def clear_chat():
        memory.clear()
        return [], ''

    send_btn.click(
        fn=respond,
        inputs=[msg_input, chatbot, groq_key_input],
        outputs=[chatbot, msg_input]
    )
    msg_input.submit(
        fn=respond,
        inputs=[msg_input, chatbot, groq_key_input],
        outputs=[chatbot, msg_input]
    )
    clear_btn.click(fn=clear_chat, outputs=[chatbot, msg_input])

demo.launch(share=True, debug=True)
print('✅ Dr. AI is running!')